# Cross-Region Carbon Intensity and Forecast Comparison

*A common-timeline study of Northern, Western, and Southern India*

**Regions:** IN-NO, IN-WE, and IN-SO<br>
**Common period:** 1 January 2017 00:00 UTC to 2 September 2026 15:00 UTC<br>
**Common test horizon:** Final 168 hours<br>
**Contributors:** Tejeshwar C D R (IN-NO), Shantharam P (IN-SO), and Harish Krishna B (IN-WE)

This notebook compares regional levels, seasonality, renewable relationships, model accuracy, residual quality, and the value of routing flexible electricity use toward the cleanest region.

## 1. Setup and aligned data

*Load the three complete datasets and verify timestamp equality*

A cross-region result is valid only when every region is observed at exactly the same hours.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.grid_pulse_analysis import load_region, validate_region

REGIONS = ["IN-NO", "IN-WE", "IN-SO"]
REGION_NAMES = {"IN-NO": "Northern India", "IN-WE": "Western India", "IN-SO": "Southern India"}
CONTRIBUTORS = {"IN-NO": "Tejeshwar C D R", "IN-SO": "Shantharam P", "IN-WE": "Harish Krishna B"}
PLOT_CONFIG = {
    "responsive": True,
    "displaylogo": False,
    "scrollZoom": True,
    "modeBarButtonsToRemove": ["lasso2d", "select2d"],
}

def show_chart(fig, height=None):
    '''Render a chart cleanly in Jupyter, VS Code, and presentation mode.'''
    fig.update_layout(
        template="plotly_white",
        autosize=True,
        width=None,
        height=height or fig.layout.height or 520,
        margin=dict(l=70, r=45, t=90, b=85),
        font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
        title=dict(x=0.01, xanchor="left", font=dict(size=20)),
        hoverlabel=dict(font_size=13),
    )
    fig.update_xaxes(automargin=True, showgrid=True, gridcolor="#e5e7eb")
    fig.update_yaxes(automargin=True, showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    fig.show(config=PLOT_CONFIG)

frames = {region: load_region(region, ROOT) for region in REGIONS}
checks = pd.DataFrame([{"Region": region, **validate_region(frame)} for region, frame in frames.items()])
same_index = all(frames[REGIONS[0]].index.equals(frames[region].index) for region in REGIONS[1:])
display(checks)
display(Markdown(
    f"All three datasets have **{len(frames['IN-NO']):,} hours**, and exact timestamp equality is "
    f"**{same_index}**. Every regional comparison below is therefore like-for-like."
))

,Region,Rows,Start,End,Duplicate timestamps,Missing target values,Missing hours,Extra hours
0,IN-NO,84760,2017-01-01 00:00:00+00:00,2026-09-02 15:00:00+00:00,0,0,0,0
1,IN-WE,84760,2017-01-01 00:00:00+00:00,2026-09-02 15:00:00+00:00,0,0,0,0
2,IN-SO,84760,2017-01-01 00:00:00+00:00,2026-09-02 15:00:00+00:00,0,0,0,0


All three datasets have **84,760 hours**, and exact timestamp equality is **True**. Every regional comparison below is therefore like-for-like.

## 2. Regional level and long-term trend

*Compare daily carbon intensity and its one-year rolling mean*

Hover and zoom to inspect any period. The thicker rolling line helps separate long-term movement from day-to-day noise.

In [2]:
daily_parts = []
for region, frame in frames.items():
    daily = frame["carbon"].resample("D").mean().to_frame("Carbon intensity")
    daily["365-day mean"] = daily["Carbon intensity"].rolling(365, min_periods=180).mean()
    daily["Region"] = region
    daily_parts.append(daily.reset_index())
daily_all = pd.concat(daily_parts, ignore_index=True)
fig = px.line(daily_all, x="datetime", y="365-day mean", color="Region",
              title="Regional carbon intensity: 365-day rolling averages")
fig.update_yaxes(title="gCO₂eq/kWh")
show_chart(fig)

level_rows = []
for region, frame in frames.items():
    first = frame["carbon"].iloc[:365 * 24].mean()
    last = frame["carbon"].iloc[-365 * 24:].mean()
    level_rows.append({"Region": region, "Full-period mean": frame["carbon"].mean(),
                       "First-year mean": first, "Latest-year mean": last,
                       "Change": last - first, "Renewable mean (%)": frame["renewable_pct"].mean()})
levels = pd.DataFrame(level_rows).sort_values("Full-period mean")
display(levels.round(2))
cleanest = levels.iloc[0]["Region"]
dirtiest = levels.iloc[-1]["Region"]
display(Markdown(
    f"**{cleanest}** has the lowest full-period average, while **{dirtiest}** has the highest. "
    "The change column shows that regional decarbonization has not happened at the same speed, so one national average would hide useful differences."
))

,Region,Full-period mean,First-year mean,Latest-year mean,Change,Renewable mean (%)
2,IN-SO,548.38,633.36,518.42,-114.94,27.05
0,IN-NO,553.77,607.30,520.78,-86.52,29.81
1,IN-WE,670.93,707.54,639.13,-68.41,14.69


**IN-SO** has the lowest full-period average, while **IN-WE** has the highest. The change column shows that regional decarbonization has not happened at the same speed, so one national average would hide useful differences.

## 3. Daily shape and renewable relationship

*Compare the typical hour and the strength of the carbon-renewable link*

The daily profile indicates when flexible work may be cleaner within each region. Correlation shows how closely renewable share explains carbon intensity.

In [3]:
profile_parts, relation_rows = [], []
for region, frame in frames.items():
    profile = frame.groupby(frame.index.hour)["carbon"].mean().rename("Carbon intensity").reset_index()
    profile["Region"] = region
    profile_parts.append(profile)
    relation_rows.append({"Region": region,
                          "Carbon-renewable correlation": frame["carbon"].corr(frame["renewable_pct"]),
                          "Average daily swing": profile["Carbon intensity"].max() - profile["Carbon intensity"].min(),
                          "Cleanest UTC hour": int(profile.loc[profile["Carbon intensity"].idxmin(), "datetime"])})
profiles = pd.concat(profile_parts, ignore_index=True).rename(columns={"datetime": "UTC hour"})
fig = px.line(profiles, x="UTC hour", y="Carbon intensity", color="Region", markers=True,
              title="Typical hourly carbon-intensity profile")
show_chart(fig)
relations = pd.DataFrame(relation_rows)
display(relations.round(3))
display(Markdown(
    "The cleanest hour is not identical in every region. Flexible jobs should therefore use "
    "regional forecasts instead of one fixed national schedule. All three correlations are negative, confirming that "
    "higher renewable share usually means lower carbon intensity."
))

,Region,Carbon-renewable correlation,Average daily swing,Cleanest UTC hour
0,IN-NO,-0.995,97.346,6
1,IN-WE,-0.988,65.256,7
2,IN-SO,-0.980,94.742,8


The cleanest hour is not identical in every region. Flexible jobs should therefore use regional forecasts instead of one fixed national schedule. All three correlations are negative, confirming that higher renewable share usually means lower carbon intensity.

## 4. Forecast accuracy across regions

*Compare models fitted with identical settings and scored on the same final week*

Lower RMSE is better. The table and chart make both the best model and the size of regional forecast difficulty visible.

In [4]:
metrics = pd.read_csv(ROOT / "results" / "forecast_metrics.csv")
winners = metrics.loc[metrics.groupby("Region")["RMSE"].idxmin()].sort_values("RMSE")
display(winners[["Region", "Model", "RMSE", "MAE", "MAPE (%)", "R²",
                 "RMSE improvement vs best baseline (%)"]].round(3))
fig = px.bar(metrics, x="RMSE", y="Model", color="Region", barmode="group",
             title="Test RMSE by model and region", height=720)
show_chart(fig)
easiest = winners.iloc[0]
hardest = winners.iloc[-1]
display(Markdown(
    f"**{easiest['Region']}** has the smallest winning RMSE (**{easiest['RMSE']:.2f}**), while "
    f"**{hardest['Region']}** has the largest (**{hardest['RMSE']:.2f}**). Model choice is regional: the same family "
    "does not automatically dominate everywhere."
))

,Region,Model,RMSE,MAE,MAPE (%),R²,RMSE improvement vs best baseline (%)
0,IN-NO,"ARIMA stationary: (2,0,1) on Δ24",11.957,9.387,2.384,0.981,7.608
22,IN-SO,"SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",17.211,13.706,3.117,0.956,11.070
11,IN-WE,"ARIMA stationary: (2,0,1) on Δ24",19.608,16.157,2.863,0.934,14.009


**IN-NO** has the smallest winning RMSE (**11.96**), while **IN-WE** has the largest (**19.61**). Model choice is regional: the same family does not automatically dominate everywhere.

## 5. The four required ARIMA/SARIMA combinations

*Separate the effects of seasonality and differencing in each region*

The heatmap uses one common color scale. Darker/lower values indicate a more accurate forecast.

In [5]:
required_names = [
    "ARIMA stationary: (2,0,1) on Δ24", "ARIMA non-stationary: (2,1,1)",
    "SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",
    "SARIMA non-stationary: (1,0,1)(1,1,1,24)"]
required = metrics[metrics["Model"].isin(required_names)]
matrix = required.pivot(index="Model", columns="Region", values="RMSE").reindex(required_names)
fig = px.imshow(matrix, text_auto=".1f", aspect="auto", color_continuous_scale="RdYlGn_r",
                title="RMSE of the four required combinations")
show_chart(fig)
required_winners = required.loc[required.groupby("Region")["RMSE"].idxmin()][["Region", "Model", "RMSE", "MAE", "R²"]]
display(required_winners.round(3))
display(Markdown(
    "Explicit lag-24 differencing is strongest in every region's best required specification. "
    "SARIMA wins in the South, while the simpler ARIMA wins in the North and West. A daily cycle can be strong even "
    "when extra SARIMA parameters do not improve a particular test week."
))

,Region,Model,RMSE,MAE,R²
0,IN-NO,"ARIMA stationary: (2,0,1) on Δ24",11.957,9.387,0.981
22,IN-SO,"SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",17.211,13.706,0.956
11,IN-WE,"ARIMA stationary: (2,0,1) on Δ24",19.608,16.157,0.934


Explicit lag-24 differencing is strongest in every region's best required specification. SARIMA wins in the South, while the simpler ARIMA wins in the North and West. A daily cycle can be strong even when extra SARIMA parameters do not improve a particular test week.

## 6. Residual quality across regions

*Check whether the winning forecast errors are unbiased and pattern-free*

Small mean residuals are preferable. A Ljung-Box p-value below 0.05 means repeated lag-24 structure remains.

In [6]:
diagnostics = pd.read_csv(ROOT / "results" / "residual_diagnostics.csv")
winner_keys = winners[["Region", "Model"]]
winner_diagnostics = winner_keys.merge(diagnostics, on=["Region", "Model"], how="left")
display(winner_diagnostics.round(4))
display(Markdown(
    "None of the regional winners should be treated as perfect. When the white-noise check fails, "
    "some predictable daily structure remains. Regular retraining, rolling validation, and prediction intervals are "
    "appropriate next steps."
))

,Region,Model,Mean residual (bias),Residual SD,Ljung-Box p (lag 24),White-noise at 5%?,Jarque-Bera p,Rank
0,IN-NO,"ARIMA stationary: (2,0,1) on Δ24",-5.0965,10.8483,0.0,No,0.2322,1
1,IN-SO,"SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",4.3212,16.7093,0.0,No,0.7224,1
2,IN-WE,"ARIMA stationary: (2,0,1) on Δ24",-9.7572,17.0586,0.0,No,0.0011,1


None of the regional winners should be treated as perfect. When the white-noise check fails, some predictable daily structure remains. Regular retraining, rolling validation, and prediction intervals are appropriate next steps.

## 7. How to leverage regional differences

*Estimate the opportunity from routing flexible demand to the cleanest available region*

This is a descriptive upper-bound calculation. At each historical hour, it chooses the region with the lowest observed carbon intensity and compares that result with always using one fixed region. Real deployment would also need capacity, latency, cost, reliability, and data-residency limits.

In [7]:
carbon_wide = pd.concat({region: frame["carbon"] for region, frame in frames.items()}, axis=1)
cleanest_each_hour = carbon_wide.min(axis=1)
chosen_region = carbon_wide.idxmin(axis=1)
routing = pd.DataFrame({
    "Strategy": [*REGIONS, "Hourly cleanest region"],
    "Average carbon intensity": [*[carbon_wide[r].mean() for r in REGIONS], cleanest_each_hour.mean()],
})
routing["Reduction vs IN-WE (%)"] = (carbon_wide["IN-WE"].mean() - routing["Average carbon intensity"]) / carbon_wide["IN-WE"].mean() * 100
display(routing.round(2))
shares = chosen_region.value_counts(normalize=True).reindex(REGIONS, fill_value=0).mul(100).rename("Hours selected (%)").reset_index().rename(columns={"index": "Region"})
display(shares.round(2))
fig = px.bar(shares, x="Region", y="Hours selected (%)", color="Region",
             title="How often each region had the lowest observed carbon intensity")
show_chart(fig)
routing_value = routing.loc[routing["Strategy"] == "Hourly cleanest region", "Reduction vs IN-WE (%)"].iloc[0]
display(Markdown(
    f"Perfect historical routing lowers average carbon intensity by **{routing_value:.1f}%** "
    "relative to always using IN-WE. The selected-region shares show that the opportunity comes from diversity: "
    "regional renewable patterns do not peak at exactly the same time. A real scheduler should use forecasts, not future actual values."
))

,Strategy,Average carbon intensity,Reduction vs IN-WE (%)
0,IN-NO,553.77,17.46
1,IN-WE,670.93,0.00
2,IN-SO,548.38,18.27
3,Hourly cleanest region,526.70,21.50


,Region,Hours selected (%)
0,IN-NO,46.47
1,IN-WE,0.00
2,IN-SO,53.53


Perfect historical routing lowers average carbon intensity by **21.5%** relative to always using IN-WE. The selected-region shares show that the opportunity comes from diversity: regional renewable patterns do not peak at exactly the same time. A real scheduler should use forecasts, not future actual values.

## 8. Cross-region conclusion

*Combine the descriptive, forecast, and operational findings*

- The three regions differ materially in average carbon intensity, long-term change, daily shape, and forecast difficulty.
- Stationary lag-24 input is consistently valuable, but the best seasonal complexity differs by region.
- Northern India is the easiest of the three to forecast in the shared final week. Western and Southern India have larger winning errors.
- Regional diversity can be useful. Flexible compute, charging, or industrial work can be shifted across both time and geography when operational constraints allow.
- A practical system should forecast each region separately, rank expected carbon intensity, include uncertainty, and apply constraints for cost, latency, capacity, reliability, and policy.

**Research answer:** One national model is not enough. Region-specific forecasting plus constrained carbon-aware routing can use differences in renewable timing and carbon intensity more effectively than a fixed regional schedule.

## 9. Individual contribution and reproducibility

*Record the integrated work and make the comparison repeatable*

Regional ownership is recorded as follows:

| Region | Contributor | Primary evidence |
|---|---|---|
| IN-NO | Tejeshwar C D R | `IN_NO/main.ipynb` |
| IN-SO | Shantharam P | `IN_SO/main.ipynb` |
| IN-WE | Harish Krishna B | `IN_WE/main.ipynb` |

The shared pipeline, comparison design, and final cross-region interpretation are documented in `src/grid_pulse_analysis.py`, `scripts/`, `results/`, and this notebook.

Run `python scripts/run_models.py` to refresh the saved metrics and forecasts, then run this notebook from top to bottom. See the regional notebooks for full assumptions and limitations.